# $Q$ Validator Introduction

## 1. Introduction

### 1.1 General Overview

This notebook introduces the $Q$-field validator developed for active Beris-Edwards systems. The goal of the validator is to take processed simulation outputs, reconstruct the relevant tensor and flow quantities, and then test whether the simulated data are consistent with the target continuum equation.

In practice, the validator is designed to answer questions such as:

- whether the simulated $Q$ dynamics match the intended Beris-Edwards equation,
- whether the recovered coefficients agree with the solver input parameters,

In principle, the same framework could also be used to test whether experimental data are consistent with this equation and to estimate the corresponding parameters. However, that use case would first require a separate denoising step. The present validator does not include such a denoising pipeline. Because its primary design goal is solver validation, it assumes that the input data are already sufficiently smooth.

The present repository focuses first on validating the $Q$-tensor equation itself, before moving on to a separate validation of the velocity equation.

### 1.2 Method: SINDy

The main identification tool used here is [SINDy](https://pysindy.readthedocs.io/en/latest/), short for Sparse Identification of Nonlinear Dynamics. The basic idea is to construct a library of candidate terms that may appear in the governing equation, evaluate those terms from data, and then use sparse regression to determine which terms are actually needed and what their coefficients are.

Using the PySINDy package is straightforward. It can be installed with:

```bash
pip install pysindy
```

To understand the basic spirit of this validator, we could consider the simple diffusion equation $\partial_t\phi=D\nabla^2\phi$. Starting from numerical snapshots of $\phi$, we first sample a set of spatial points and time levels. At each sampled point, finite differences are used to calculate $\partial_t\phi$ and $\nabla^2\phi$ from the numerical solution. These values form a linear regression problem, $y_i=D x_i$, where $y_i=(\partial_t\phi)_i$ and $x_i=(\nabla^2\phi)_i$. Fitting this relation over all sampled points gives the diffusion coefficient $D$. The validator follows the same procedure, except that the regression contains more candidate terms and therefore more columns in the regression matrix. All five independent components of $Q$ are included together in the regression.

For the $Q$-equation, this means we assemble a candidate library containing terms such as material advection, vorticity coupling, flow-alignment terms, bulk free-energy terms, and elastic terms. SINDy then determines whether the data support the expected equation structure and whether the fitted coefficients match the intended physical parameters.

This approach is useful because it converts equation validation into a concrete regression problem with measurable outputs such as recovered coefficients, $R^2$, and relative residual.

### 1.3 Target Equation

The $Q$ validator tests an equation whose left-hand side is simply

$$
\partial_t Q.
$$

The right-hand side is built from the following candidate terms currently implemented in the packaged $Q$ validator:

| Meaning | Formula | Coefficient |
| --- | --- | --- |
| material advection | $(\mathbf{u}\cdot\nabla)Q$ | $\lambda_c$ |
| corotational derivative | $\Omega\,Q - Q\,\Omega$ | $\lambda_r$ |
| plain strain term | $E$ | $\lambda_1$ |
| strain-Q interaction | $E\,Q + Q\,E$ | $\lambda_2$ |
| nonlinear flow-alignment term | $(Q:E)\,Q$ | $\lambda_3$ |
| linear bulk term | $Q$ | $-\Gamma a_2$ |
| quadratic bulk term | $Q^2$ | $-\Gamma a_3$ |
| cubic bulk term | $\operatorname{tr}(Q^2)Q$ | $-\Gamma a_4$ |
| one-constant elastic term | $\nabla^2 Q$ | $\Gamma L_1$ |

Here $Q$ is the traceless symmetric nematic tensor order parameter, $\mathbf{u}$ is the velocity field, $E = (\nabla \mathbf{u} + (\nabla \mathbf{u})^T)/2$ is the strain-rate tensor, and $\Omega = (\nabla \mathbf{u} - (\nabla \mathbf{u})^T)/2$ is the vorticity tensor. The final recovered parameters in the output are $\lambda_c$, $\lambda_r$, $\lambda_1$, $\lambda_2$, $\lambda_3$, $a_2$, $a_3$, $a_4$, and $L_1$.

**All of these terms will go through symmetric-traceless projection, which performed as:** $\mathcal{P}_{\mathrm{ST}}(A) = \frac{1}{2}(A + A^T) - \frac{1}{3}\operatorname{tr}(A)I$.

Because the molecular field $H$ (the potential force given by free energy) enters the equation as $\partial_t Q = ... + \Gamma H$, $\Gamma$ is degenerate with the free-energy coefficients: the regression identifies the products $\Gamma a_2$, $\Gamma a_3$, $\Gamma a_4$, and $\Gamma L_1$, rather than the free-energy coefficients alone. Therefore, the value of $\Gamma$ must be provided manually to recover the actual free-energy coefficients. The default value is $\Gamma=1$.

Future versions will include the $L_2$ and $L_3$ elastic free-energy terms to move beyond the one-constant approximation.

The material-advection term is discretized using a third-order upwind stencil. All other spatial derivatives, including those used to construct $E$, $\Omega$, and $\nabla^2Q$, are evaluated using centered finite differences.


## 2. Usage

### 2.1 Generate and Prepare the Simulation Data

Run the solver once and save both the $Q$ field and the velocity field at every time step. The number of saved time steps can be chosen according to the needs of the validation. When possible, use a sufficiently complex state as the initial condition so that the candidate terms exhibit distinct spatial structures and are less likely to be strongly collinear.

After the simulation, convert the numerical solutions into two NumPy files with the following names and layouts:

| File name | Array shape | Component order |
|---|---|---|
| `q.npy` | $(T, N_x, N_y, N_z, 5)$ | $(Q_{xx}, Q_{xy}, Q_{xz}, Q_{yy}, Q_{yz})$ |
| `velocity.npy` | $(T, N_x, N_y, N_z, 3)$ | $(u_x, u_y, u_z)$ |

Here, $T$ is the number of saved time levels. The time levels must be stored in chronological order, and both files must use the same $(T, N_x, N_y, N_z)$ layout. Because $Q$ is symmetric and traceless, the validator reconstructs $Q_{yx}$, $Q_{zx}$, $Q_{zy}$, and $Q_{zz}=-(Q_{xx}+Q_{yy})$ from the five stored components.


### 2.2 Run the Validator and Set the Parameters

Pass the two prepared files to `discover_q_from_processed_npy(...)`. Set `dt` to the time interval between two saved fields, `spacings` to the grid spacings $(\Delta x, \Delta y, \Delta z)$, and `gamma` to the rotational mobility $\Gamma$ used by the solver. Other parameters use their default values in this minimal example.


In [1]:
from pathlib import Path
import sys

# Locate the example directory whether Jupyter starts here or at the repository root.
example_dir = Path.cwd()
if example_dir.name != "example":
    example_dir = example_dir / "example"

validator_root = example_dir.parent
checkq_root = validator_root.parent / "CheckQ_SINDy"
checkq_src = checkq_root / "src"
# Make the packaged validator and its source library importable.
for import_path in (checkq_root, checkq_src):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from app import discover_q_from_processed_npy

# result = discover_q_from_processed_npy(
#     q_npy_path="q.npy",
#     velocity_npy_path="velocity.npy",
#     dt=1.0,
#     spacings=(1.0, 1.0, 1.0),
#     gamma=1.0,
# )


### 2.3 Ludwig Example

[Ludwig](https://ludwig.epcc.ed.ac.uk/) is an open-source parallel lattice-Boltzmann code developed largely at the University of Edinburgh for the simulation of complex fluids. In addition to Newtonian fluids, Ludwig supports free-energy models for systems including binary fluids, polar gels, and liquid crystals.

Here, the packaged validator is applied to $Q$ and velocity fields generated by Ludwig. Because Ludwig is an independent solver, successfully recovering its intended $Q$ equation also provides a separate check of the correctness of the validator itself.


In [2]:
# Run Q-equation discovery on the processed Ludwig fields.
ludwig_data_dir = example_dir / "data" / "ludwig_checkpoint_dense"

ludwig_result = discover_q_from_processed_npy(
    q_npy_path=ludwig_data_dir / "cache_q_stack.npy",
    velocity_npy_path=ludwig_data_dir / "cache_u_stack.npy",
    dt=1.0,
    spacings=(1.0, 1.0, 1.0),
    gamma=2.0,
)


In [3]:
ludwig_result


{'coefficients': {'lambda_r': 1.0008890214524326,
  'lambda_1': 0.6662766030274542,
  'lambda_2': 1.0045474006248962,
  'lambda_3': -2.002953801430402,
  'a2': -0.0003323436830255028,
  'a3': -0.030859452602501797,
  'a4': 0.030868289403489308,
  'L1': 0.004112658301150009},
 'input_parameters': {'q_npy_path': 'D:\\Document\\GitHub\\ActiveBE_Validator\\example\\data\\ludwig_checkpoint_dense\\cache_q_stack.npy',
  'velocity_npy_path': 'D:\\Document\\GitHub\\ActiveBE_Validator\\example\\data\\ludwig_checkpoint_dense\\cache_u_stack.npy',
  'dt': 1.0,
  'spacings': (1.0, 1.0, 1.0),
  'time_discretization': 'two_point',
  'velocity_time_level': 'n+1',
  'q_time_level': 'n',
  'sample_step': 8,
  'threshold': 1e-08,
  'regularization_strength': 1e-12,
  'is_normalize_columns': True,
  'interior_strip': 3,
  'gamma': 2.0,
  'is_verbose': False,
  'result_output_path': None},
 'metrics': {'r2': 0.9999074657343845,
  'relative_residual': 0.009614481554594562,
  'rmse': 1.5831939118947857e-06}}

### 2.4 Understanding the Results

The validator returns a dictionary containing three sections:

- `coefficients` contains the recovered physical coefficients $\lambda_c$, $\lambda_r$, $\lambda_1$, $\lambda_2$, $\lambda_3$, $a_2$, $a_3$, $a_4$, and $L_1$.
- `input_parameters` stores the parameter values supplied to the validator so that the regression can be reproduced. The meaning of each input parameter is explained in the next section.
- `metrics` contains `r2`, `relative_residual`, and `rmse`, which quantify how closely the fitted right-hand side reproduces the measured time derivative $\partial_tQ$.

Let $y_i$ denote the sampled components of $\partial_tQ$, let $\hat{y}_i$ denote the corresponding prediction from the fitted equation, and define the residual as $r_i=y_i-\hat{y}_i$. For a total of $N$ scalar samples, the metrics are calculated as follows:

| Metric | Calculation | Meaning |
| --- | --- | --- |
| `r2` | $R^2=1-\dfrac{\sum_{i=1}^{N}r_i^2}{\sum_{i=1}^{N}(y_i-\bar{y})^2}$ | The fraction of the variation in $\partial_tQ$ explained by the fitted equation. A value close to $1$ indicates a strong fit, while a negative value means that the fit is worse than predicting the mean $\bar{y}$. |
| `relative_residual` | $\dfrac{\lVert y-\hat{y}\rVert_2}{\lVert y\rVert_2}$ | The residual magnitude normalized by the target magnitude. A value of $0$ represents exact recovery; smaller values indicate a better fit. |
| `rmse` | $\sqrt{\dfrac{1}{N}\sum_{i=1}^{N}r_i^2}$ | The root-mean-square error per scalar sample. It has the same units as $\partial_tQ$ and therefore depends on the physical scale of the data. |


### 2.5 Input Parameters

The input parameters of `discover_q_from_processed_npy(...)` are:

| Parameter | Default | Meaning |
| --- | --- | --- |
| `q_npy_path` | Required | Path to the NumPy file containing the five independent components of $Q$ with shape $(T,N_x,N_y,N_z,5)$ and component order $(Q_{xx},Q_{xy},Q_{xz},Q_{yy},Q_{yz})$. |
| `velocity_npy_path` | Required | Path to the NumPy file containing the velocity field with shape $(T,N_x,N_y,N_z,3)$ and component order $(u_x,u_y,u_z)$. |
| `dt` | `1.0` | Time interval between two consecutive saved fields. This value is used to calculate $\partial_tQ$. |
| `spacings` | `(1.0, 1.0, 1.0)` | Grid spacings $(\Delta x,\Delta y,\Delta z)$ used by all spatial finite-difference operators. |
| `time_discretization` | `"two_point"` | Time-difference formula for $\partial_tQ$. `"two_point"` uses $(Q^{n+1}-Q^n)/\Delta t$, while `"central"` uses $(Q^{n+1}-Q^{n-1})/(2\Delta t)$. The default is `"two_point"` because numerical solutions of the $Q$ equation are often advanced using a simple Euler method or one of its variants, allowing the corresponding discrete update to be validated directly. For experimental data, the centered difference is generally the better choice. |
| `velocity_time_level` | `"n+1"` | Time level of the velocity-dependent terms on the right-hand side. It may be set to `"n"` or `"n+1"`. |
| `q_time_level` | `"n"` | Time level of the $Q$-dependent terms on the right-hand side. It may be set to `"n"` or `"n+1"`. Together, `velocity_time_level` and `q_time_level` account for the update order used by the solver. For example, suppose the solver starts from $u^n$ and $Q^n$, first computes $u^{n+1}$ using $u^n$ and $Q^n$, and then computes $Q^{n+1}$ using $u^{n+1}$ and $Q^n$. In this case, set `velocity_time_level="n+1"` and `q_time_level="n"`. |
| `sample_step` | `8` | Spatial stride between sampled regression points. An integer applies the same stride in all three directions; a three-entry sequence sets separate strides. The derivative stencils still use neighboring values on the original fine grid. This parameter controls spatial sampling only; the current implementation uses all valid time levels. |
| `threshold` | `1e-8` | Sparsity threshold used by sequentially thresholded least squares. Candidate terms whose fitted contribution falls below the threshold are removed from the model. |
| `regularization_strength` | `1e-12` | Ridge regularization strength used during sparse regression. Larger values apply stronger coefficient regularization. |
| `is_normalize_columns` | `True` | Whether to normalize the candidate-library columns before sparse regression so terms with different numerical scales can be compared more consistently. |
| `interior_strip` | `3` | Number of fine-grid points excluded from each side of every spatial boundary before regression points are sampled. |
| `gamma` | `1.0` | Rotational mobility $\Gamma$. It is required to separate the recovered products $\Gamma a_2$, $\Gamma a_3$, $\Gamma a_4$, and $\Gamma L_1$ into the reported free-energy coefficients. |
| `is_verbose` | `False` | Whether to print progress messages while the candidate terms are constructed and fitted. |
| `result_output_path` | `None` | Optional result-storage path. `None` disables file output. A directory produces `q_discovery_result.json` inside that directory, while a path ending in `.json` uses the specified file name. Missing parent directories are created automatically. |


## 3. Other Examples

### 3.1 ANS: Exact Recovery

The following example validates data generated by a solver which uses the exact same stencil with the validator. For this dataset, the validator hence recovers the $Q$ equation with $R^2=1$.


In [2]:
# Run Q-equation discovery on the processed ANS fields.
ans_data_dir = example_dir / "data" / "ans"

ans_result = discover_q_from_processed_npy(
    q_npy_path=ans_data_dir / "cache_q_stack.npy",
    velocity_npy_path=ans_data_dir / "cache_u_stack.npy",
    dt=1.0,
    spacings=(1.0, 1.0, 1.0),
    gamma=2.0,
)


In [3]:
ans_result

{'coefficients': {'lambda_r': 0.9999999999999949,
  'lambda_1': 0.666666666666664,
  'lambda_2': 0.9999999999999948,
  'lambda_3': -2.0000000000000124,
  'a2': -0.0003336333333333153,
  'a3': -0.031027900000000046,
  'a4': 0.031027899999999945,
  'L1': 0.004099669999999982},
 'input_parameters': {'q_npy_path': 'D:\\Document\\GitHub\\ActiveBE_Validator\\example\\data\\ans\\cache_q_stack.npy',
  'velocity_npy_path': 'D:\\Document\\GitHub\\ActiveBE_Validator\\example\\data\\ans\\cache_u_stack.npy',
  'dt': 1.0,
  'spacings': (1.0, 1.0, 1.0),
  'time_discretization': 'two_point',
  'velocity_time_level': 'n+1',
  'q_time_level': 'n',
  'sample_step': 8,
  'threshold': 1e-08,
  'regularization_strength': 1e-12,
  'is_normalize_columns': True,
  'interior_strip': 3,
  'gamma': 2.0,
  'is_verbose': False,
  'result_output_path': None},
 'metrics': {'r2': 1.0,
  'relative_residual': 4.149245124746974e-14,
  'rmse': 7.183950794056156e-18}}